# Kvasir-SEG Attention U-Net Colab

Run this notebook on Google Colab with GPU enabled: Runtime -> Change runtime type -> T4 GPU.

In [ ]:
!nvidia-smi

## 1. Mount Google Drive and clone the repository

All training outputs are written to Google Drive so they survive Colab runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/kvasir_colab_outputs'
!mkdir -p "{OUTPUT_DIR}"
print('Outputs will be saved to:', OUTPUT_DIR)

In [ ]:
%cd /content
!rm -rf cnn_learning
!git clone https://github.com/qrkks/cnn_learning.git
%cd cnn_learning/4_medical_segmentation
!pip install -r requirements.txt

## 2. Download Kvasir-SEG

In [ ]:
!mkdir -p data
!rm -f data/kvasir-seg.zip
!wget --no-check-certificate -O data/kvasir-seg.zip https://datasets.simula.no/downloads/kvasir-seg.zip
!python -c "import zipfile; assert zipfile.is_zipfile('data/kvasir-seg.zip'), 'Downloaded file is not a valid zip.'"
!unzip -q -o data/kvasir-seg.zip -d data
!find data -maxdepth 2 -type d

If the extracted folder is not `data/Kvasir-SEG`, edit `configs/colab.yaml` so `data_root` points to the folder containing `images/` and `masks/`.

In [ ]:
!python src/train.py --config configs/colab.yaml --output-dir "{OUTPUT_DIR}"

In [ ]:
!python src/evaluate.py --config configs/colab.yaml --checkpoint "{OUTPUT_DIR}/best_model.pth" --output-dir "{OUTPUT_DIR}" --num-visuals 8

In [ ]:
import json
import pandas as pd

with open(f'{OUTPUT_DIR}/metrics_val.json', 'r', encoding='utf-8') as f:
    metrics = json.load(f)

history = pd.read_csv(f'{OUTPUT_DIR}/history.csv')
print(metrics)
history.tail()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['epoch'], history['train_loss'], label='train loss')
axes[0].plot(history['epoch'], history['val_loss'], label='val loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history['epoch'], history['train_dice'], label='train Dice')
axes[1].plot(history['epoch'], history['val_dice'], label='val Dice')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice')
axes[1].legend()

fig.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=160)
plt.show()

In [ ]:
from IPython.display import Image, display
display(Image(f'{OUTPUT_DIR}/predictions/prediction_000.png'))

In [ ]:
import shutil
from google.colab import files

zip_base = '/content/kvasir_colab_outputs'
shutil.make_archive(zip_base, 'zip', root_dir='/content/drive/MyDrive', base_dir='kvasir_colab_outputs')
files.download(zip_base + '.zip')